# 02 — Rollout: avaliação no LiberoEnv

Carrega o melhor checkpoint (selecionado por `val_recon_z0`) e roda a política no ambiente com temporal ensembling.

In [ ]:
# --- Setup: clonar (limpo) + instalar em modo editável ---
# Repo privado? Guarde um token no Secrets do Colab e use:
#   from google.colab import userdata; token = userdata.get("GH_TOKEN")
#   !git clone https://{token}@github.com/rafaelheydt/act-lang.git /content/act-lang
#
# %cd /content ANTES do rm -rf: se uma execução anterior desta célula deixou
# o shell dentro de /content/act-lang, apagar essa pasta com o shell "sentado"
# nela quebra o cwd do processo (erros "getcwd: cannot access parent
# directories") e derruba até o git clone seguinte.
%cd /content
!rm -rf /content/act-lang
!git clone https://github.com/rafaelheydt/act-lang.git /content/act-lang
%cd /content/act-lang
!pip install -q -e . "lerobot[libero]"

import sys
# "configs/" fica FORA de src/ de propósito (configs de experimento editáveis
# sem reinstalar nada) -- por isso não é abrangido pelo pip install -e .
# Alguns kernels IPython não resolvem import a partir do cwd dinamicamente,
# então o insert explícito é a forma confiável de tornar "from configs...."
# importável, independente de como aquele kernel específico se comporta.
if "/content/act-lang" not in sys.path:
    sys.path.insert(0, "/content/act-lang")

import os
os.environ["MUJOCO_GL"] = "egl"  # headless (Colab)

# SEM autoreload: o IPython pré-instalado no Colab (pinado em 7.34.0 pelo
# pacote google-colab) quebra com o autoreload no runtime atual, e forçar o
# upgrade do IPython quebra drive.mount()/exibição de vídeo em troca.
#
# IMPORTANTE: depois desta célula rodar pela primeira vez em CADA runtime
# novo, faça Runtime > Restart session antes de continuar -- o Python só lê
# o registro do "pip install -e ." (arquivo .pth) na inicialização do
# interpretador, não em tempo real. Sem o restart, "import act_lang" falha
# com ModuleNotFoundError mesmo com tudo instalado corretamente.
#
# Reload cirúrgico sem reiniciar, se preferir (depois do primeiro restart):
#   import importlib, act_lang.models.act
#   importlib.reload(act_lang.models.act)

In [ ]:
import numpy as np
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from libero.libero import benchmark
from lerobot.envs.libero import LiberoEnv

from configs.libero_single_task import CONFIG as cfg
from act_lang.data.normalize import MinMaxNormalizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

meta = LeRobotDatasetMetadata("lerobot/libero")
state_norm = MinMaxNormalizer.from_lerobot_stats(meta.stats, "observation.state").to(device)
action_norm = MinMaxNormalizer.from_lerobot_stats(meta.stats, "action").to(device)

# task_id da instrução treinada
(TAREFA,) = cfg["task_texts"]
task_suite = benchmark.get_benchmark_dict()[cfg["task_suite_name"]]()
task_id = next(
    i for i in range(len(task_suite.tasks))
    if task_suite.get_task(i).language == TAREFA
)
print(f"task_id={task_id} | \"{TAREFA}\"")

In [ ]:
# --- Carregar melhor checkpoint ---
from pathlib import Path
from act_lang.models.act import ACT
from act_lang.models.backbone import freeze_batchnorm
from act_lang.training.checkpoints import load_checkpoint

checkpoint_dir = Path("/content/drive/MyDrive") / cfg["experiment_name"]
from google.colab import drive
drive.mount("/content/drive")

model = ACT(
    action_dim=cfg["action_dim"], state_dim=cfg["state_dim"],
    d_model=cfg["d_model"], latent_dim=cfg["latent_dim"],
    chunk_size=cfg["chunk_size"], n_cameras=cfg["n_cameras"],
    n_encoder_layers=cfg["n_encoder_layers"], n_decoder_layers=cfg["n_decoder_layers"],
    n_heads=cfg["n_heads"], dropout=cfg["dropout"], pretrained_backbone=False,
)
if cfg["freeze_bn"]:
    freeze_batchnorm(model.vision_backbone)  # mesma estrutura do treino
model = model.to(device)

best = sorted(checkpoint_dir.glob("best_epoch*.pt"))[-1]  # ou aponte o arquivo à mão
print(f"carregando: {best.name}")
next_epoch, _ = load_checkpoint(best, model, device=device)
model.eval()
print(f"checkpoint da época {next_epoch - 1}")

In [ ]:
# --- Rollout ---
from act_lang.eval.rollout_libero import rollout_libero

env = LiberoEnv(
    task_suite=task_suite, task_id=task_id,
    task_suite_name=cfg["task_suite_name"],
    control_mode="relative", render_mode="rgb_array",
)

results = rollout_libero(
    model, env, state_norm, action_norm, device,
    n_episodes=cfg["rollout_n_episodes"], m=cfg["rollout_m"],
    max_steps=cfg["rollout_max_steps"], video_dir="/content/libero_rollouts",
    video_fps=int(meta.fps),
)
env.close()

success_rate = np.mean([r["success"] for r in results])
print(f"\nTaxa de sucesso: {success_rate:.1%}")

In [ ]:
# --- Ver um episódio ---
from IPython.display import Video, display
tag = "sucesso" if results[0]["success"] else "falha"
display(Video(f"/content/libero_rollouts/ep00_{tag}.mp4", embed=True, width=700))